# WASH4/WASH5/ENTR1 (Jurkat) — Temporary Assembly Notebook

Adapted from `XL_MOPLC_pipeline.ipynb`. This is a **temporary, one-off notebook**
for integrating a new WASHC4–WASHC5–ENTR1 AF3 trimer prediction (3 samples,
Jurkat XL-MS data) into the existing `assembled_complex` project, and re-running
the assembly for:

**WASHC1, WASHC2C, WASHC3, WASHC4, WASHC5, ENTR1** (chains A, B, C, D, E, F).

FKBP15 (chain G) is intentionally excluded from this sub-assembly.

---

### What this notebook does

1. Copy the non-FKBP15 dimer/trimer predictions already processed in the
   original `assembled_complex/rewrited_pdbs` into this project.
2. Rewrite the *new* WASHC4-WASHC5-ENTR1 AF3 trimer predictions
   (`afx_pred/washc4_washc5_entr1_0/1/2`) to standardized PDB + confidence JSON.
3. Merge the new Jurkat XL-MS crosslink evidence (WASHC5-ENTR1 ×3 new,
   WASHC4-ENTR1 ×1 new) into `ucrosslinks.csv`.
4. Build a 6-chain `useqs.csv` and `network.csv` (10 edges, including the new
   WASHC4-ENTR1 edge supported by the new crosslink).
5. Split trimers into dimers, select representative structures.
6. Run MCTS assembly and inspect the result.

> **Note:** cells are written but **not executed** in this file — run them
> interactively and check each step's output before proceeding to the next.


---
## Global Configuration

Paths reflect the already-reorganized `wash4_wash5_entr1_jurkat` project
directory (see conversation history for how it was reorganized:
`afx_pred/`, `jsons/`, `xl_ms_data/`, `chains.csv`, `WASH.fasta`,
`assembled_complex/`).


In [2]:
import os
import sys
import shutil
import json as _json
import pandas as pd

# ==============================================================
# Project paths
# ==============================================================
BASE_DIR = r"N:\08_NK_structure_prediction\data\WASH_complex\wash4_wash5_entr1_jurkat"

# --- Raw AF3 inputs/outputs for the new WASHC4-WASHC5-ENTR1 trimer ---
AF3_JSON_DIR = f"{BASE_DIR}\\jsons"
AF_PRED_DIR  = f"{BASE_DIR}\\afx_pred"
XL_MS_DIR    = f"{BASE_DIR}\\xl_ms_data"
NEW_XL_XLSX  = f"{XL_MS_DIR}\\wash4_wash5_entr1_jurkat.xlsx"

# --- Shared gene/chain/sequence files (copied from assembled_complex) ---
CHAINS_CSV = f"{BASE_DIR}\\chains.csv"
FASTA_PATH = f"{BASE_DIR}\\WASH.fasta"

# --- This project's assembly working directory ---
ASSEMBLED_DIR    = f"{BASE_DIR}\\assembled_complex"
NETWORK_CSV      = f"{ASSEMBLED_DIR}\\network.csv"
USEQS_CSV        = f"{ASSEMBLED_DIR}\\useqs.csv"
UCROSSLINKS_CSV  = f"{ASSEMBLED_DIR}\\ucrosslinks.csv"
REWRITED_PDB_DIR = f"{ASSEMBLED_DIR}\\rewrited_pdbs"
PAIRS_DIR        = f"{ASSEMBLED_DIR}\\pairs"
OUTPUT_DIR       = f"{ASSEMBLED_DIR}\\output"

# --- Original (source) assembled_complex project to pull existing pairs from ---
OLD_ASSEMBLED_DIR     = r"N:\08_NK_structure_prediction\data\WASH_complex\assembled_complex"
OLD_CHAINS_CSV        = f"{OLD_ASSEMBLED_DIR}\\chains.csv"
OLD_FASTA_PATH        = f"{OLD_ASSEMBLED_DIR}\\WASH.fasta"
OLD_USEQS_CSV         = f"{OLD_ASSEMBLED_DIR}\\useqs.csv"
OLD_UCROSSLINKS_CSV   = f"{OLD_ASSEMBLED_DIR}\\ucrosslinks.csv"
OLD_REWRITED_PDB_DIR  = f"{OLD_ASSEMBLED_DIR}\\rewrited_pdbs"

# --- Assembly parameters ---
CROSSLINKER_LENGTH = 40   # Ca-Ca cutoff (Angstrom) for DSS/BS3-type crosslinker

# --- Target complex: WASHC1, WASHC2C, WASHC3, WASHC4, WASHC5, ENTR1 (no FKBP15) ---
TARGET_CHAINS = ["A", "B", "C", "D", "E", "F"]
GENE_TO_CHAIN = {
    "WASHC1": "A", "WASHC2C": "B", "WASHC3": "C",
    "WASHC4": "D", "WASHC5": "E", "ENTR1": "F",
}

# Make the pipeline modules importable
sys.path.insert(0, r"N:\08_NK_structure_prediction\XL_MOPLC\XL_complex_structure\src")

print(f"BASE_DIR = {BASE_DIR}")
print(f"OLD_ASSEMBLED_DIR = {OLD_ASSEMBLED_DIR}")


BASE_DIR = N:\08_NK_structure_prediction\data\WASH_complex\wash4_wash5_entr1_jurkat
OLD_ASSEMBLED_DIR = N:\08_NK_structure_prediction\data\WASH_complex\assembled_complex


---
## Step A · Copy Existing (non-FKBP15) Dimer/Trimer Predictions

Copy every `rewrited_pdbs` subfolder from the original `assembled_complex`
project that does **not** involve FKBP15 (chain G). These already cover all
pairwise/triple combinations of WASHC1, WASHC2C, WASHC3, WASHC4, WASHC5,
ENTR1 except the brand-new WASHC4-WASHC5-ENTR1 trimer handled in Step B.


In [3]:
os.makedirs(REWRITED_PDB_DIR, exist_ok=True)

copied, skipped = 0, 0
for name in sorted(os.listdir(OLD_REWRITED_PDB_DIR)):
    if "fkbp15" in name.lower():
        continue  # exclude FKBP15-containing predictions
    src = os.path.join(OLD_REWRITED_PDB_DIR, name)
    dst = os.path.join(REWRITED_PDB_DIR, name)
    if not os.path.isdir(src):
        continue
    if os.path.isdir(dst) and os.listdir(dst):
        skipped += 1
        continue
    shutil.copytree(src, dst)
    copied += 1

print(f"Copied {copied} folders, skipped {skipped} already-present folders.")


Copied 0 folders, skipped 51 already-present folders.


---
## Step B · Rewrite the New WASHC4-WASHC5-ENTR1 AF3 Trimer Predictions

Convert the 3 new AF3 samples in `afx_pred/washc4_washc5_entr1_0/1/2` to
standardized PDB + confidence JSON with renamed chains (D, E, F), matching
`chains.csv`. Output merges into the same `rewrited_pdbs` folder used in
Step A.


In [4]:
from complex_assembly.rewrite_af_files import (
    rewrite_af_cif_structure,
    rewrite_af_score_file,
    split_trimer_to_dimers,
    select_most_central_pdb,
)
import complex_assembly.mcts as mcts

rewrite_af_cif_structure(
    af_pred_folder=AF_PRED_DIR,
    chains_df_path=CHAINS_CSV,
    output_folder=REWRITED_PDB_DIR,
)
rewrite_af_score_file(
    af_pred_folder=AF_PRED_DIR,
    chains_df_path=CHAINS_CSV,
    output_folder=REWRITED_PDB_DIR,
)
print("New trimer rewritten into:", REWRITED_PDB_DIR)


Gene -> Chain mapping: {'washc1': 'A', 'washc2c': 'B', 'washc3': 'C', 'washc4': 'D', 'washc5': 'E', 'entr1': 'F', 'fkbp15': 'G'}
Folder: washc4_washc5_entr1_0, mapping: {'A': 'D', 'B': 'E', 'C': 'F'}


c:\Users\zhang.zehong\AppData\Local\anaconda3\envs\xl_complex_structure\Lib\site-packages\Bio\PDB\Entity.py:197: BiopythonWarning: The id `D` is already used for a sibling of this entity. Changing id from `A` to `D` might create access inconsistencies to children of the parent entity.
  warnings.warn(
c:\Users\zhang.zehong\AppData\Local\anaconda3\envs\xl_complex_structure\Lib\site-packages\Bio\PDB\Entity.py:197: BiopythonWarning: The id `E` is already used for a sibling of this entity. Changing id from `B` to `E` might create access inconsistencies to children of the parent entity.
  warnings.warn(


Saved rewritten PDB: N:\08_NK_structure_prediction\data\WASH_complex\wash4_wash5_entr1_jurkat\assembled_complex\rewrited_pdbs\washc4_washc5_entr1_0\washc4_washc5_entr1_0.pdb
Folder: washc4_washc5_entr1_1, mapping: {'A': 'D', 'B': 'E', 'C': 'F'}


c:\Users\zhang.zehong\AppData\Local\anaconda3\envs\xl_complex_structure\Lib\site-packages\Bio\PDB\Entity.py:197: BiopythonWarning: The id `D` is already used for a sibling of this entity. Changing id from `A` to `D` might create access inconsistencies to children of the parent entity.
  warnings.warn(
c:\Users\zhang.zehong\AppData\Local\anaconda3\envs\xl_complex_structure\Lib\site-packages\Bio\PDB\Entity.py:197: BiopythonWarning: The id `E` is already used for a sibling of this entity. Changing id from `B` to `E` might create access inconsistencies to children of the parent entity.
  warnings.warn(
c:\Users\zhang.zehong\AppData\Local\anaconda3\envs\xl_complex_structure\Lib\site-packages\Bio\PDB\Entity.py:197: BiopythonWarning: The id `F` is already used for a sibling of this entity. Changing id from `C` to `F` might create access inconsistencies to children of the parent entity.
  warnings.warn(


Saved rewritten PDB: N:\08_NK_structure_prediction\data\WASH_complex\wash4_wash5_entr1_jurkat\assembled_complex\rewrited_pdbs\washc4_washc5_entr1_1\washc4_washc5_entr1_1.pdb
Folder: washc4_washc5_entr1_2, mapping: {'A': 'D', 'B': 'E', 'C': 'F'}


c:\Users\zhang.zehong\AppData\Local\anaconda3\envs\xl_complex_structure\Lib\site-packages\Bio\PDB\Entity.py:197: BiopythonWarning: The id `D` is already used for a sibling of this entity. Changing id from `A` to `D` might create access inconsistencies to children of the parent entity.
  warnings.warn(
c:\Users\zhang.zehong\AppData\Local\anaconda3\envs\xl_complex_structure\Lib\site-packages\Bio\PDB\Entity.py:197: BiopythonWarning: The id `E` is already used for a sibling of this entity. Changing id from `B` to `E` might create access inconsistencies to children of the parent entity.
  warnings.warn(


Saved rewritten PDB: N:\08_NK_structure_prediction\data\WASH_complex\wash4_wash5_entr1_jurkat\assembled_complex\rewrited_pdbs\washc4_washc5_entr1_2\washc4_washc5_entr1_2.pdb
Done!
Gene -> Chain mapping: {'washc1': 'A', 'washc2c': 'B', 'washc3': 'C', 'washc4': 'D', 'washc5': 'E', 'entr1': 'F', 'fkbp15': 'G'}
Folder: washc4_washc5_entr1_0, mapping: {'A': 'D', 'B': 'E', 'C': 'F'}
Folder: washc4_washc5_entr1_1, mapping: {'A': 'D', 'B': 'E', 'C': 'F'}
Folder: washc4_washc5_entr1_2, mapping: {'A': 'D', 'B': 'E', 'C': 'F'}
Done!
New trimer rewritten into: N:\08_NK_structure_prediction\data\WASH_complex\wash4_wash5_entr1_jurkat\assembled_complex\rewrited_pdbs


---
## Step C · Merge New XL-MS Crosslink Evidence

Start from the original `ucrosslinks.csv`, drop every row touching FKBP15
(chain G), then append the new Jurkat crosslinks from
`xl_ms_data/wash4_wash5_entr1_jurkat.xlsx`:

| WASH subunit | WASH residue | ENTR1 residue |
|---|---|---|
| WASHC5 | 481 | 118 |
| WASHC5 | 468 | 118 |
| WASHC5 | 468 | 124 |
| WASHC5 | 468 | 134 |
| WASHC4 | 1089 | 134 |

(WASHC5-481–ENTR1-118 already existed in the original file; duplicates are
dropped.)


In [11]:
xl = pd.read_csv(OLD_UCROSSLINKS_CSV)
xl_no_fkbp15 = xl[(xl["ChainA"] != "G") & (xl["ChainB"] != "G")].reset_index(drop=True)

new_xl = pd.read_excel(NEW_XL_XLSX)
new_rows = []
for _, row in new_xl.iterrows():
    a_chain = GENE_TO_CHAIN[row["WASH subunit"]]
    new_rows.append({
        "ChainA": a_chain, "ResidueA": int(row["WASH residue"]),
        "ChainB": "F",     "ResidueB": int(row["ENTR1 residue"]),
    })
new_xl_df = pd.DataFrame(new_rows)

ucrosslinks = pd.concat([xl_no_fkbp15, new_xl_df], ignore_index=True)
ucrosslinks = ucrosslinks.drop_duplicates(
    subset=["ChainA", "ResidueA", "ChainB", "ResidueB"]
).reset_index(drop=True)

os.makedirs(ASSEMBLED_DIR, exist_ok=True)
ucrosslinks.to_csv(UCROSSLINKS_CSV, index=False)
print(f"Saved {len(ucrosslinks)} crosslinks -> {UCROSSLINKS_CSV}")
ucrosslinks.tail(6)


Saved 53 crosslinks -> N:\08_NK_structure_prediction\data\WASH_complex\wash4_wash5_entr1_jurkat\assembled_complex\ucrosslinks.csv


,ChainA,ResidueA,ChainB,ResidueB
47,D,1089,D,1147
48,F,116,F,134
49,E,468,F,118
50,E,468,F,124
51,E,468,F,134
52,D,1089,F,134


---
## Step D · Build the 6-Chain `useqs.csv`

Drop the FKBP15 (chain G) row from the original `useqs.csv`.


In [6]:
useqs = pd.read_csv(OLD_USEQS_CSV)
useqs6 = useqs[useqs["Chain"] != "G"].reset_index(drop=True)
useqs6.to_csv(USEQS_CSV, index=False)
print("Chains kept:", useqs6["Chain"].tolist())
useqs6


Chains kept: ['A', 'B', 'C', 'D', 'E', 'F']


,Chain,Useq,Sequence
0,A,WASHC1,MTPVRMQHSLAGQTYAVPFIQPDLRREEAVQQMADALQYLQKVSGD...
1,B,WASHC2C,MMNRTTPDQELVPASEPVWERPWSVEEIRRSSQSWSLAADAGLLQF...
2,C,WASHC3,MDEDGLPLMGSGIDLTKVPAIQQKRTVAFLNQFVVHTVQFLNRFST...
3,D,WASHC4,MAVETLSPDWEFDRVDDGSQKIHAEVQLKNYGKFLEEYTSQLRRIE...
4,E,WASHC5,MLDFLAENNLCGQAILRIVSCGNAIIAELLRLSEFIPAVFRLKDRA...
5,F,ENTR1,MSGYQRRPGATPLSRARSLAIPDAPAFYERRSCLPQLNCERPHGRD...


---
## Step E · Build `network.csv`

Edges = confirmed PPI pairs among the 6 target proteins
(`binary_pairs_in_ppi.csv`, excluding FKBP15) **plus** the new WASHC4-ENTR1
edge supported by the new crosslink (D-1089 &ndash; F-134).
`Source` = sorted chain-pair string = the pair folder name under `pairs/`.


In [ ]:
# edges = [
#     ("A", "B"),  # WASHC1-WASHC2C
#     ("A", "E"),  # WASHC1-WASHC5
#     ("A", "D"),  # WASHC1-WASHC4
#     ("A", "C"),  # WASHC1-WASHC3
#     ("B", "C"),  # WASHC2C-WASHC3
#     ("B", "D"),  # WASHC2C-WASHC4
#     ("C", "D"),  # WASHC3-WASHC4
#     ("D", "E"),  # WASHC4-WASHC5
#     ("E", "F"),  # WASHC5-ENTR1
#     ("D", "F"),  # WASHC4-ENTR1 (new, from Jurkat XL-MS)
# ]

# network_df = pd.DataFrame([
#     {"Chain1": c1, "Chain2": c2, "Source": "".join(sorted([c1, c2]))}
#     for c1, c2 in edges
# ])
# network_df.to_csv(NETWORK_CSV, index=False)
# print(f"Saved {len(network_df)} edges -> {NETWORK_CSV}")
# network_df


Saved 10 edges -> N:\08_NK_structure_prediction\data\WASH_complex\wash4_wash5_entr1_jurkat\assembled_complex\network.csv


,Chain1,Chain2,Source
0,A,B,AB
1,A,E,AE
2,A,D,AD
3,A,C,AC
4,B,C,BC
5,B,D,BD
6,C,D,CD
7,D,E,DE
8,E,F,EF
9,D,F,DF


---
## Step F · Split Trimers into Dimers

Decomposes every 2-/3-chain prediction in `rewrited_pdbs` (Steps A+B) into
standardized dimer pairs under `pairs/`. Progress is checkpointed, so this
cell is safe to re-run if interrupted.


In [12]:
split_trimer_to_dimers(
    rewrited_pdb_folder=REWRITED_PDB_DIR,
    output_folder=PAIRS_DIR,
)
print("Dimer pairs written to:", PAIRS_DIR)


[SKIP] washc1_washc2c_0 already processed
[SKIP] washc1_washc2c_1 already processed
[SKIP] washc1_washc2c_2 already processed
[SKIP] washc1_washc2c_washc3_0 already processed
[SKIP] washc1_washc2c_washc3_1 already processed
[SKIP] washc1_washc2c_washc3_2 already processed
[SKIP] washc1_washc3_washc4_0 already processed
[SKIP] washc1_washc3_washc4_1 already processed
[SKIP] washc1_washc3_washc4_2 already processed
[SKIP] washc1_washc3_washc5_0 already processed
[SKIP] washc1_washc3_washc5_1 already processed
[SKIP] washc1_washc3_washc5_2 already processed
[SKIP] washc1_washc4_0 already processed
[SKIP] washc1_washc4_1 already processed
[SKIP] washc1_washc4_2 already processed
[SKIP] washc1_washc5_0 already processed
[SKIP] washc1_washc5_1 already processed
[SKIP] washc1_washc5_2 already processed
[SKIP] washc1_washc5_entr1_0 already processed
[SKIP] washc1_washc5_entr1_1 already processed
[SKIP] washc1_washc5_entr1_2 already processed
[SKIP] washc2c_washc3_0 already processed
[SKIP] was

---
## Step G · Select Representative Dimer Structures

For each pair, picks the most central structure (lowest mean C-alpha RMSD to
all other candidates) as the representative for assembly.

> **Network-mount note:** if `pairs/` lives on a filesystem/mount that does
> not support `chmod` (some SMB/network shares), `shutil.copy`'s metadata
> step will raise `PermissionError`. The monkeypatch below makes that a
> no-op without touching `rewrite_af_files.py`; remove it if not needed.


In [13]:
# Work around chmod-unsupported network mounts (no-op if not needed).
shutil.copymode = lambda *a, **k: None

select_most_central_pdb(PAIRS_DIR)
print("Representative dimers selected in:", PAIRS_DIR)


[SKIP] AB already finalized
[SKIP] AC already finalized
[SKIP] AD already finalized
[SKIP] AE already finalized
[SKIP] AF already finalized
[SKIP] BC already finalized
[SKIP] BD already finalized
[SKIP] BE already finalized
[SKIP] CD already finalized
[SKIP] CE already finalized
[SKIP] DE already finalized
[SKIP] DF already finalized
[SKIP] EF already finalized
Done
Representative dimers selected in: N:\08_NK_structure_prediction\data\WASH_complex\wash4_wash5_entr1_jurkat\assembled_complex\pairs


---
## Step H · MCTS-Based Complex Assembly

Assembles WASHC1, WASHC2C, WASHC3, WASHC4, WASHC5, ENTR1 (chains A-F) using
the 10-edge network from Step E.

> **Bug note:** the shared `complex_assembly/mcts.py` had a bug where
> `best_path()` aborted the entire search on the first steric-clash
> rejection instead of retrying another candidate edge. This has already
> been fixed in-place in `mcts.py` (see the file's `best_path` method) —
> no workaround is needed here, just documented for awareness.


In [14]:
import argparse

os.makedirs(OUTPUT_DIR, exist_ok=True)

args = argparse.Namespace(
    network            = NETWORK_CSV,
    pairdir             = PAIRS_DIR,
    useqs               = USEQS_CSV,
    ucrosslinks         = UCROSSLINKS_CSV,
    outdir              = OUTPUT_DIR,
    crosslinker_length  = CROSSLINKER_LENGTH,
)

mcts.main(args)
print(f"\nAssembly completed. Results in: {OUTPUT_DIR}")


2026-07-04 14:17:25 | INFO | Loading input files...
2026-07-04 14:17:25 | INFO | Network: 9 edges | Chains: 6 | Crosslinks: 53 | Crosslinker length: 40 Å
2026-07-04 14:17:25 | INFO | Running MCTS with 6 root candidates...
2026-07-04 14:17:25 | INFO | 
2026-07-04 14:17:25 | INFO | === MCTS root = A ===
2026-07-04 14:17:27 | INFO | A -> C
2026-07-04 14:17:34 | INFO | A -> D
2026-07-04 14:17:40 | INFO | A -> E
2026-07-04 14:17:43 | INFO | A -> B
2026-07-04 14:17:47 | INFO | A -> E -> F
2026-07-04 14:17:50 | INFO | A -> E -> D
2026-07-04 14:17:52 | INFO | A -> E -> C
2026-07-04 14:17:56 | INFO | A -> E -> D
2026-07-04 14:18:00 | INFO | A -> E -> B
2026-07-04 14:18:03 | INFO | A -> E -> F -> D
2026-07-04 14:18:06 | INFO | A -> E -> F -> D
2026-07-04 14:18:09 | INFO | A -> E -> F -> C
2026-07-04 14:18:12 | INFO | A -> E -> F -> D
2026-07-04 14:18:16 | INFO | A -> E -> F -> B
2026-07-04 14:18:19 | INFO | A -> E -> F -> D -> C
2026-07-04 14:18:22 | INFO | A -> E -> F -> D -> B
2026-07-04 14:18


Assembly completed. Results in: N:\08_NK_structure_prediction\data\WASH_complex\wash4_wash5_entr1_jurkat\assembled_complex\output


---
## Step I · Inspect Results

List the assembled complex PDB, the optimal assembly path, and confirm all
6 chains are present.


In [15]:
best_complex_pdb = os.path.join(OUTPUT_DIR, "best_complex.pdb")
optimal_path_csv = os.path.join(OUTPUT_DIR, "optimal_path.csv")

print("best_complex.pdb exists:", os.path.exists(best_complex_pdb))
print("optimal_path.csv exists:", os.path.exists(optimal_path_csv))

if os.path.exists(optimal_path_csv):
    display(pd.read_csv(optimal_path_csv))

if os.path.exists(best_complex_pdb):
    chains_found = set()
    with open(best_complex_pdb) as fh:
        for line in fh:
            if line.startswith("ATOM"):
                chains_found.add(line[21])
    print("Chains in best_complex.pdb:", sorted(chains_found))
    assert chains_found == set(TARGET_CHAINS), "Missing chains in final assembly!"


best_complex.pdb exists: True
optimal_path.csv exists: True


,Chain,Edge_chain,Source
0,F,E,EF
1,A,B,AB
2,E,D,DE
3,D,C,CD
4,C,B,BC


Chains in best_complex.pdb: ['A', 'B', 'C', 'D', 'E', 'F']
